# Election OCR — Preprocess + EDA Notebook

## Imports and paths

In [1]:
from pathlib import Path
import sys

import duckdb
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "election_ocr").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / "src"
if SRC_PATH.exists() and str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))


def resolve_project_path(path) -> Path:
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


try:
    from election_ocr.config import settings
    SOURCE_DB = resolve_project_path(settings.gold_db)
    CLEAN_DB = resolve_project_path(settings.gold_db.parent / "elections.clean.duckdb")
    OUT_DIR = resolve_project_path(settings.data_root / "analysis_exports")
except Exception:
    SOURCE_DB = resolve_project_path("data/gold/elections.duckdb")
    CLEAN_DB = resolve_project_path("data/gold/elections.clean.duckdb")
    OUT_DIR = resolve_project_path("data/analysis_exports")

TABLES_TO_PREPROCESS = [
    "constituency_stations",
    "candidate_votes",
    "partylist_stations",
    "partylist_votes",
]

print("Source DB:", SOURCE_DB)
print("Clean DB :", CLEAN_DB)
print("Output   :", OUT_DIR)

Source DB: /Users/wutthichod/Documents/dsde/final_project/data/gold/elections.duckdb
Clean DB : /Users/wutthichod/Documents/dsde/final_project/data/gold/elections.clean.duckdb
Output   : /Users/wutthichod/Documents/dsde/final_project/data/analysis_exports


## 2. Helper functions — preprocessing


In [2]:
def preprocess_table(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    print(f"Preprocessing table: {table_name}")

    print("Before preprocessing:")
    print(df.shape)

    print("Missing values before:")
    print(df.isna().sum().sort_values(ascending=False))

    df = df.copy()

    # drop unused columns
    drop_cols = [col for col in ["issues", "warnings"] if col in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)
        print(f"Dropped columns: {drop_cols}")

    # detect and drop duplicated data
    dedup_cols = []
    for col in df.columns:
        sample = df[col].dropna()

        if len(sample) == 0:
            dedup_cols.append(col)
            continue

        first_value = sample.iloc[0]

        if isinstance(first_value, (list, tuple)):
            continue

        try:
            hash(first_value)
            dedup_cols.append(col)
        except TypeError:
            continue

    before_dup = len(df)
    if dedup_cols:
        df = df.drop_duplicates(subset=dedup_cols)
    print(f"Removed duplicated rows: {before_dup - len(df)}")

    # fill nan
    numeric_cols = df.select_dtypes(include="number").columns
    if len(numeric_cols) > 0:
        df[numeric_cols] = df[numeric_cols].fillna(0)

    text_cols = df.select_dtypes(include=["object", "string"]).columns
    if len(text_cols) > 0:
        df[text_cols] = df[text_cols].fillna("")

    bool_cols = df.select_dtypes(include="bool").columns
    if len(bool_cols) > 0:
        df[bool_cols] = df[bool_cols].fillna(False)

    # drop all zero numeric rows
    if len(numeric_cols) > 0:
        before_zero = len(df)
        df = df.loc[(df[numeric_cols] != 0).any(axis=1)]
        print(f"Removed all-zero numeric rows: {before_zero - len(df)}")

    print("Missing values after:")
    print(df.isna().sum().sort_values(ascending=False))

    print("After preprocessing:")
    print(df.shape)

    return df


## Preprocessing

In [3]:
def run_preprocess():
    if not Path(SOURCE_DB).exists():
        raise FileNotFoundError(
            f"Source DB not found: {SOURCE_DB}\n"
        )

    print(f"Source DB: {SOURCE_DB}")
    print(f"Clean DB : {CLEAN_DB}")

    CLEAN_DB.parent.mkdir(parents=True, exist_ok=True)

    source_con = duckdb.connect(str(SOURCE_DB))
    clean_con = duckdb.connect(str(CLEAN_DB))

    tables = source_con.sql("SHOW TABLES").df()
    print("Available tables:")
    display(tables)

    table_names = tables.iloc[:, 0].tolist()
    created_tables = []

    for table_name in TABLES_TO_PREPROCESS:
        if table_name not in table_names:
            print(f"Skipping missing table: {table_name}")
            continue

        print(f"Loading table: {table_name}")
        df = source_con.sql(f"SELECT * FROM {table_name}").df()
        clean_df = preprocess_table(df, table_name)

        clean_con.register("tmp_df", clean_df)
        clean_con.execute(f"DROP TABLE IF EXISTS {table_name}")
        clean_con.execute(
            f"CREATE TABLE {table_name} AS SELECT * FROM tmp_df"
        )
        clean_con.unregister("tmp_df")

        created_tables.append(table_name)

    source_con.close()
    clean_con.close()

    print("\nPreprocessing completed")
    print("Created clean DB:", CLEAN_DB)
    print("Created tables:")
    for table in created_tables:
        print(f"- {table}")

run_preprocess()


Source DB: /Users/wutthichod/Documents/dsde/final_project/data/gold/elections.duckdb
Clean DB : /Users/wutthichod/Documents/dsde/final_project/data/gold/elections.clean.duckdb
Available tables:


,name
0,candidate_votes
1,constituency_stations
2,partylist_stations
3,partylist_votes


Loading table: constituency_stations
Preprocessing table: constituency_stations
Before preprocessing:
(302, 21)
Missing values before:
station_id           302
sha256                 0
ballots_allocated      0
issues                 0
validation_passed      0
total_votes            0
ballots_remaining      0
ballots_no_vote        0
ballots_invalid        0
ballots_valid          0
ballots_used           0
voters_present         0
voting_type            0
voters_registered      0
election_date          0
polling_station        0
constituency           0
changwat               0
amphoe                 0
tambon                 0
warnings               0
dtype: int64
Dropped columns: ['issues', 'warnings']
Removed duplicated rows: 0
Removed all-zero numeric rows: 0
Missing values after:
sha256               0
voters_present       0
total_votes          0
ballots_remaining    0
ballots_no_vote      0
ballots_invalid      0
ballots_valid        0
ballots_used         0
ballots_allocated    

In [4]:
if not Path(CLEAN_DB).exists():
    raise FileNotFoundError(
        f"DuckDB file not found: {CLEAN_DB}\n"
    )

preview_con = duckdb.connect(str(CLEAN_DB), read_only=True)
for table_name in TABLES_TO_PREPROCESS:
    print(f"\nPreview: {table_name}")
    display(preview_con.sql(f"SELECT * FROM {table_name}").df().head(10))
preview_con.close()



Preview: constituency_stations


,sha256,voting_type,station_id,tambon,amphoe,changwat,constituency,polling_station,election_date,voters_registered,voters_present,ballots_allocated,ballots_used,ballots_valid,ballots_invalid,ballots_no_vote,ballots_remaining,total_votes,validation_passed
0,03e9067a33b6cb9f39d8447cecedc3cdf1ce2e2945cf2c...,election_day,,ทุ่งช้าง,เมืองเชียงใหม่,ลำปาง,1,10,2026-02-28,84,60,840,609,512,34,63,291,512,False
1,04064c739bb309b8835c6af157d6ad1cf94144b35994c8...,election_day,,เมืองตาล,ทางเข้า การ,ลำปาง,1,6,2026-02-04,500,700,500,300,305,200,200,100,305,False
2,049c9f74ec9652571bf1e6400aa1f9f19df0aef7e90bbc...,election_day,,มอแซ็ค,เมืองลำปาง,ลำปาง,1,4,2026-02-28,730,531,680,531,464,15,52,149,464,False
3,06e957a45b33ef48362893252df010d1b3c2050801ce3b...,election_day,,สำราญ,เชื่อมลำปลา,ลำปาง,1,22,2026-02-28,528,356,460,356,320,8,28,104,320,False
4,1c11a72209b392cbcecaff9d94339586c50d3d44561e86...,election_day,,เชียงตาล,ซังฮัตร,ลำปาง,1,11,2026-02-28,40,260,380,260,230,176,14,120,230,False
5,3390e60c0797905b6ac7b85c7ad90635e2119c9f685f18...,election_day,,ต้น ธงชัย,เมือง,ลำปาง,1,4,2026-02-28,651,450,620,450,389,26,35,170,389,False
6,3618f3b07935ef129a9acf610a703b457c86568dbefd7a...,election_day,,น่อแป๋ว,เมือง,ลำปาง,1,12,2026-02-28,172,136,180,136,122,2,12,44,122,False
7,37267c6d8b6d7f433f1467e2b52876496cb72afd2887c0...,advance,,ไม่ระบุ,ไม่ระบุ,ลำปาง,1,1478,2026-02-08,760,760,760,760,692,0,68,0,692,True
8,39b8ed1299b3339fd9f8d43d198a42f662314f2e4bac0a...,election_day,,คันธรชัย,เมืองลำปาง,ลำปาง,1,19,2026-02-28,631,449,580,449,377,14,58,131,377,True
9,3b6dffaa888ccd263e36e97f94ce5a6fa0b916a9de58b9...,election_day,,หนองหล่ม,ห้องพักอาศัย,ลำปาง,1,6,2026-02-28,340,265,240,265,244,3,18,75,244,True



Preview: candidate_votes


,sha256,voting_type,candidate_number,candidate_name,party,votes
0,017e407e54aeef30e3b729e9d86d0b1faa90f573ed843f...,election_day,3,นายชวนิต จันทรสุรินทร์,พรรคภูมิใจไทย,23
1,017e407e54aeef30e3b729e9d86d0b1faa90f573ed843f...,election_day,6,ร.ต.อ. ประสิทธิ์ ทิวงศ์ษา,พรรคเศรษฐกิจ,4
2,030889866b59d4d3aad16eaeaf17d033a92ccc2cb84ce6...,election_day,5,นายนริศ แสงบุญเรือง,พรรคไทยก้าวใหม่,5
3,030889866b59d4d3aad16eaeaf17d033a92ccc2cb84ce6...,election_day,9,นายก้องเกียรติ มาทา,พรรครวมไทยสร้างชาติ,10
4,03e9067a33b6cb9f39d8447cecedc3cdf1ce2e2945cf2c...,election_day,2,นายบุญเชิด พรมศร,พรรคโอกาสใหม่,0
5,04064c739bb309b8835c6af157d6ad1cf94144b35994c8...,election_day,1,นายอธิวัฒน์ ศรีไชยยานุพันธ์,พรรคคล้ายธรรม,9
6,04d0a0fa76437268a0165e269b910250aa3d20fa56cd43...,election_day,7,นางสาวอมลยา เจนตวนิชย์,พรรคประชาธิปัตย์,10
7,0614618f5745dd81ef08f5ff37cfdd2270e7ec65a2bc0e...,election_day,1,นายอธิวัฒน์ ศรีไชยยานุพันธ์,พรรคคล้ายธรรม,0
8,06e957a45b33ef48362893252df010d1b3c2050801ce3b...,election_day,1,นายอธิวัฒน์ ศรีไชยยานุพันธ์,พรรคคล้ายธรรม,1
9,0abb5a3d5ba816eff3555ebefde09ba608ad7570a965c4...,election_day,5,นายนริศ แสงบุญเรือง,พรรคไทยก้าวใหม่,2



Preview: partylist_stations


,sha256,voting_type,station_id,tambon,amphoe,changwat,constituency,polling_station,election_date,voters_registered,voters_present,ballots_allocated,ballots_used,ballots_valid,ballots_invalid,ballots_no_vote,ballots_remaining,total_votes,validation_passed
0,03e9067a33b6cb9f39d8447cecedc3cdf1ce2e2945cf2c...,election_day,,ฝั่งฝ่าย,เมืองลำปาง,ลำปาง,1,10,2026-02-28,86,60,840,609,513,51,45,291,513,False
1,04064c739bb309b8835c6af157d6ad1cf94144b35994c8...,election_day,,เมืองตาล,ทางเข้า,ลำปาง,1,6,2026-02-28,50,38,500,381,333,13,35,107,333,False
2,049c9f74ec9652571bf1e6400aa1f9f19df0aef7e90bbc...,election_day,,น่าเข้าฯ,เมืองลำปาง,ลำปาง,1,4,2026-02-28,730,531,680,531,487,13,3,149,487,False
3,06e957a45b33ef48362893252df010d1b3c2050801ce3b...,election_day,,สำราญ,เชื่อมลำปลา,ลำปาง,1,22,2026-02-28,528,356,460,356,332,87,28,104,332,False
4,1c11a72209b392cbcecaff9d94339586c50d3d44561e86...,election_day,,เลของคาว,ห้างจักร,ลำปาง,1,11,2026-02-28,490,260,380,260,239,710,11,120,239,False
5,3390e60c0797905b6ac7b85c7ad90635e2119c9f685f18...,election_day,,คันธรวย,เมือง,ลำปาง,1,4,2026-02-08,651,450,620,450,396,27,27,170,396,False
6,3618f3b07935ef129a9acf610a703b457c86568dbefd7a...,election_day,,ป้อมแพ่ง,เมือง,ลำปาง,1,12,2026-02-28,172,136,180,136,128,3,5,44,128,False
7,37267c6d8b6d7f433f1467e2b52876496cb72afd2887c0...,advance,,,,ลำปาง,1,1478,2026-02-28,760,760,760,760,727,0,33,0,727,False
8,39b8ed1299b3339fd9f8d43d198a42f662314f2e4bac0a...,election_day,,คันธรพย์,เมืองลำปาง,ลำปาง,1,19,2026-02-28,631,409,580,449,412,8,29,131,412,False
9,3b6dffaa888ccd263e36e97f94ce5a6fa0b916a9de58b9...,election_day,,หนองเหลม,หัวลังค์ครู,ลำปาง,1,2,2026-02-28,340,265,340,265,252,6,7,75,252,False



Preview: partylist_votes


,sha256,voting_type,party_number,party_name,votes
0,010c3b8ee364d964a940de6ab3d1ffee97d65cf8355745...,election_day,8,ประชาธิปไตยใหม่,8
1,010c3b8ee364d964a940de6ab3d1ffee97d65cf8355745...,election_day,13,รวมพลังประชาชน,0
2,010c3b8ee364d964a940de6ab3d1ffee97d65cf8355745...,election_day,19,สังคมประชาธิปไตยไทย,0
3,010c3b8ee364d964a940de6ab3d1ffee97d65cf8355745...,election_day,46,ประชาชน,189
4,015b04d2eb819a076b045cf6876d3236424c066ee0632d...,election_day,8,ประชาธิปไตยใหม่,800
5,015b04d2eb819a076b045cf6876d3236424c066ee0632d...,election_day,13,รวมพลังประชาชน,1
6,015b04d2eb819a076b045cf6876d3236424c066ee0632d...,election_day,19,สังคมประชาธิปไตยไทย,4
7,015b04d2eb819a076b045cf6876d3236424c066ee0632d...,election_day,46,ประชาชน,197
8,017e407e54aeef30e3b729e9d86d0b1faa90f573ed843f...,election_day,7,พลวัต,2
9,017e407e54aeef30e3b729e9d86d0b1faa90f573ed843f...,election_day,27,ประชาธิปัตย์,0


## Helper functions and Inspet data — EDA

In [5]:
def safe_divide(a: pd.Series, b: pd.Series) -> pd.Series:
    return a / b.replace(0, pd.NA)


def read_table(con: duckdb.DuckDBPyConnection, table_name: str) -> pd.DataFrame:
    tables = con.sql("SHOW TABLES").df().iloc[:, 0].tolist()
    if table_name in tables:
        return con.sql(f"SELECT * FROM {table_name}").df()
    raise ValueError(f"{table_name} does not exist. Available tables: {tables}")


def load_tables():
    if not Path(CLEAN_DB).exists():
        raise FileNotFoundError(
            f"DuckDB file not found: {CLEAN_DB}\n"
        )

    con = duckdb.connect(str(CLEAN_DB), read_only=True)
    constituency_stations = read_table(con, "constituency_stations")
    candidate_votes = read_table(con, "candidate_votes")
    partylist_stations = read_table(con, "partylist_stations")
    partylist_votes = read_table(con, "partylist_votes")
    con.close()
    return constituency_stations, candidate_votes, partylist_stations, partylist_votes


def add_station_metrics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    numeric_cols = [
        "voters_registered",
        "voters_present",
        "ballots_allocated",
        "ballots_used",
        "ballots_valid",
        "ballots_invalid",
        "ballots_no_vote",
        "ballots_remaining",
        "total_votes",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    df["turnout_rate"] = safe_divide(df["voters_present"], df["voters_registered"])
    df["valid_rate"] = safe_divide(df["ballots_valid"], df["voters_present"])
    df["invalid_rate"] = safe_divide(df["ballots_invalid"], df["voters_present"])
    df["no_vote_rate"] = safe_divide(df["ballots_no_vote"], df["voters_present"])
    return df


In [6]:
(
    constituency_stations,
    candidate_votes,
    partylist_stations,
    partylist_votes,
) = load_tables()

constituency_stations = add_station_metrics(constituency_stations)
partylist_stations = add_station_metrics(partylist_stations)

print("constituency_stations:", constituency_stations.shape)
print("candidate_votes:", candidate_votes.shape)
print("partylist_stations:", partylist_stations.shape)
print("partylist_votes:", partylist_votes.shape)

display(constituency_stations.head())
display(partylist_stations.head())
display(candidate_votes.head())
display(partylist_votes.head())

constituency_stations: (302, 23)
candidate_votes: (2745, 6)
partylist_stations: (302, 23)
partylist_votes: (16859, 5)


,sha256,voting_type,station_id,tambon,amphoe,changwat,constituency,polling_station,election_date,voters_registered,...,ballots_valid,ballots_invalid,ballots_no_vote,ballots_remaining,total_votes,validation_passed,turnout_rate,valid_rate,invalid_rate,no_vote_rate
0,03e9067a33b6cb9f39d8447cecedc3cdf1ce2e2945cf2c...,election_day,,ทุ่งช้าง,เมืองเชียงใหม่,ลำปาง,1,10,2026-02-28,84,...,512,34,63,291,512,False,0.714286,8.533333,0.566667,1.05
1,04064c739bb309b8835c6af157d6ad1cf94144b35994c8...,election_day,,เมืองตาล,ทางเข้า การ,ลำปาง,1,6,2026-02-04,500,...,305,200,200,100,305,False,1.4,0.435714,0.285714,0.285714
2,049c9f74ec9652571bf1e6400aa1f9f19df0aef7e90bbc...,election_day,,มอแซ็ค,เมืองลำปาง,ลำปาง,1,4,2026-02-28,730,...,464,15,52,149,464,False,0.727397,0.873823,0.028249,0.097928
3,06e957a45b33ef48362893252df010d1b3c2050801ce3b...,election_day,,สำราญ,เชื่อมลำปลา,ลำปาง,1,22,2026-02-28,528,...,320,8,28,104,320,False,0.674242,0.898876,0.022472,0.078652
4,1c11a72209b392cbcecaff9d94339586c50d3d44561e86...,election_day,,เชียงตาล,ซังฮัตร,ลำปาง,1,11,2026-02-28,40,...,230,176,14,120,230,False,6.5,0.884615,0.676923,0.053846


,sha256,voting_type,station_id,tambon,amphoe,changwat,constituency,polling_station,election_date,voters_registered,...,ballots_valid,ballots_invalid,ballots_no_vote,ballots_remaining,total_votes,validation_passed,turnout_rate,valid_rate,invalid_rate,no_vote_rate
0,03e9067a33b6cb9f39d8447cecedc3cdf1ce2e2945cf2c...,election_day,,ฝั่งฝ่าย,เมืองลำปาง,ลำปาง,1,10,2026-02-28,86,...,513,51,45,291,513,False,0.697674,8.55,0.85,0.75
1,04064c739bb309b8835c6af157d6ad1cf94144b35994c8...,election_day,,เมืองตาล,ทางเข้า,ลำปาง,1,6,2026-02-28,50,...,333,13,35,107,333,False,0.76,8.763158,0.342105,0.921053
2,049c9f74ec9652571bf1e6400aa1f9f19df0aef7e90bbc...,election_day,,น่าเข้าฯ,เมืองลำปาง,ลำปาง,1,4,2026-02-28,730,...,487,13,3,149,487,False,0.727397,0.917137,0.024482,0.00565
3,06e957a45b33ef48362893252df010d1b3c2050801ce3b...,election_day,,สำราญ,เชื่อมลำปลา,ลำปาง,1,22,2026-02-28,528,...,332,87,28,104,332,False,0.674242,0.932584,0.244382,0.078652
4,1c11a72209b392cbcecaff9d94339586c50d3d44561e86...,election_day,,เลของคาว,ห้างจักร,ลำปาง,1,11,2026-02-28,490,...,239,710,11,120,239,False,0.530612,0.919231,2.730769,0.042308


,sha256,voting_type,candidate_number,candidate_name,party,votes
0,017e407e54aeef30e3b729e9d86d0b1faa90f573ed843f...,election_day,3,นายชวนิต จันทรสุรินทร์,พรรคภูมิใจไทย,23
1,017e407e54aeef30e3b729e9d86d0b1faa90f573ed843f...,election_day,6,ร.ต.อ. ประสิทธิ์ ทิวงศ์ษา,พรรคเศรษฐกิจ,4
2,030889866b59d4d3aad16eaeaf17d033a92ccc2cb84ce6...,election_day,5,นายนริศ แสงบุญเรือง,พรรคไทยก้าวใหม่,5
3,030889866b59d4d3aad16eaeaf17d033a92ccc2cb84ce6...,election_day,9,นายก้องเกียรติ มาทา,พรรครวมไทยสร้างชาติ,10
4,03e9067a33b6cb9f39d8447cecedc3cdf1ce2e2945cf2c...,election_day,2,นายบุญเชิด พรมศร,พรรคโอกาสใหม่,0


,sha256,voting_type,party_number,party_name,votes
0,010c3b8ee364d964a940de6ab3d1ffee97d65cf8355745...,election_day,8,ประชาธิปไตยใหม่,8
1,010c3b8ee364d964a940de6ab3d1ffee97d65cf8355745...,election_day,13,รวมพลังประชาชน,0
2,010c3b8ee364d964a940de6ab3d1ffee97d65cf8355745...,election_day,19,สังคมประชาธิปไตยไทย,0
3,010c3b8ee364d964a940de6ab3d1ffee97d65cf8355745...,election_day,46,ประชาชน,189
4,015b04d2eb819a076b045cf6876d3236424c066ee0632d...,election_day,8,ประชาธิปไตยใหม่,800


In [7]:
def create_vote_summary(
    stations: pd.DataFrame,
    votes: pd.DataFrame,
    group_cols: list[str],
) -> pd.DataFrame:
    joined = votes.merge(
        stations[["sha256", "constituency", "ballots_valid"]],
        on="sha256",
        how="left",
    )
    joined["votes"] = pd.to_numeric(joined["votes"], errors="coerce").fillna(0)
    joined = joined[joined["votes"] <= joined["ballots_valid"]]

    summary = (
        joined.groupby(group_cols, dropna=False)
        .agg(
            total_votes=("votes", "sum"),
            avg_votes_per_station=("votes", "mean"),
            max_votes_in_station=("votes", "max"),
            station_count=("sha256", "nunique"),
        )
        .reset_index()
        .sort_values("total_votes", ascending=False)
    )
    total = summary["total_votes"].sum()
    summary["vote_share"] = summary["total_votes"] / total if total else 0
    return summary


def create_area_summary(stations: pd.DataFrame) -> pd.DataFrame:
    group_cols = [col for col in ["changwat", "constituency", "amphoe", "tambon"] if col in stations.columns]
    summary = (
        stations.groupby(group_cols, dropna=False)
        .agg(
            station_count=("sha256", "nunique"),
            voters_registered=("voters_registered", "sum"),
            voters_present=("voters_present", "sum"),
            ballots_used=("ballots_used", "sum"),
            ballots_valid=("ballots_valid", "sum"),
            ballots_invalid=("ballots_invalid", "sum"),
            ballots_no_vote=("ballots_no_vote", "sum"),
        )
        .reset_index()
    )
    summary["turnout_rate"] = safe_divide(summary["voters_present"], summary["voters_registered"])
    summary["invalid_rate"] = safe_divide(summary["ballots_invalid"], summary["voters_present"])
    summary["no_vote_rate"] = safe_divide(summary["ballots_no_vote"], summary["voters_present"])
    return summary.sort_values("station_count", ascending=False)


In [8]:
def create_vote_station_joined(
    stations: pd.DataFrame,
    votes: pd.DataFrame,
    *,
    number_col: str,
    name_col: str,
    label_col: str,
) -> pd.DataFrame:
    keep_cols = [
        "sha256", "voting_type", "changwat", "constituency", "polling_station",
        "amphoe", "tambon", "voters_registered", "voters_present", "ballots_valid",
        "ballots_invalid", "ballots_no_vote", "turnout_rate", "invalid_rate",
        "no_vote_rate", "validation_passed",
    ]
    keep_cols = [col for col in keep_cols if col in stations.columns]
    joined = votes.merge(stations[keep_cols], on="sha256", how="left")
    joined["votes"] = pd.to_numeric(joined["votes"], errors="coerce").fillna(0)
    joined = joined[joined["votes"] <= joined["ballots_valid"]]
    joined["station_vote_share"] = safe_divide(joined["votes"], joined["ballots_valid"])
    joined["choice_number"] = joined[number_col]
    joined["choice_name"] = joined[name_col]
    joined["choice_label"] = joined[label_col]
    return joined


def create_winner_by_station(vote_station: pd.DataFrame) -> pd.DataFrame:
    winner = (
        vote_station.sort_values(["sha256", "votes"], ascending=[True, False])
        .groupby("sha256")
        .head(1)
        .copy()
    )
    keep_cols = [
        "sha256", "voting_type", "changwat", "constituency", "polling_station",
        "amphoe", "tambon", "choice_number", "choice_name", "choice_label",
        "votes", "ballots_valid", "station_vote_share", "turnout_rate",
        "invalid_rate", "validation_passed",
    ]
    keep_cols = [col for col in keep_cols if col in winner.columns]
    return winner[keep_cols].sort_values("votes", ascending=False)


def create_competitive_stations(vote_station: pd.DataFrame) -> pd.DataFrame:
    ranked = vote_station.copy()
    ranked["rank_in_station"] = ranked.groupby("sha256")["votes"].rank(method="first", ascending=False)
    top2 = ranked[ranked["rank_in_station"] <= 2].copy()
    id_cols = ["sha256", "voting_type", "changwat", "constituency", "polling_station", "amphoe", "tambon"]
    id_cols = [col for col in id_cols if col in top2.columns]
    competitive = (
        top2.pivot_table(index=id_cols, columns="rank_in_station", values="votes", aggfunc="first")
        .reset_index()
        .rename(columns={1.0: "rank1_votes", 2.0: "rank2_votes"})
    )
    competitive["rank2_votes"] = competitive["rank2_votes"].fillna(0)
    competitive["vote_margin"] = competitive["rank1_votes"] - competitive["rank2_votes"]
    competitive["margin_rate"] = safe_divide(competitive["vote_margin"], competitive["rank1_votes"])
    return competitive.sort_values("vote_margin")


def create_validation_summary(stations: pd.DataFrame, table_name: str) -> pd.DataFrame:
    if "validation_passed" not in stations.columns:
        return pd.DataFrame()
    summary = stations.groupby("validation_passed", dropna=False).agg(station_count=("sha256", "nunique")).reset_index()
    summary.insert(0, "table", table_name)
    return summary

## 5. Run EDA and preview insights

In [9]:
candidate_summary = create_vote_summary(
    constituency_stations,
    candidate_votes,
    ["constituency", "candidate_number", "candidate_name", "party"],
)
partylist_summary = create_vote_summary(
    partylist_stations,
    partylist_votes,
    ["constituency", "party_number", "party_name"],
)
constituency_area_summary = create_area_summary(constituency_stations)
partylist_area_summary = create_area_summary(partylist_stations)

candidate_joined = create_vote_station_joined(
    constituency_stations,
    candidate_votes,
    number_col="candidate_number",
    name_col="candidate_name",
    label_col="party",
)
partylist_joined = create_vote_station_joined(
    partylist_stations,
    partylist_votes,
    number_col="party_number",
    name_col="party_name",
    label_col="party_name",
)

candidate_winner_by_station = create_winner_by_station(candidate_joined)
partylist_winner_by_station = create_winner_by_station(partylist_joined)
candidate_competitive_stations = create_competitive_stations(candidate_joined)
partylist_competitive_stations = create_competitive_stations(partylist_joined)
high_invalid_constituency = constituency_stations.sort_values("invalid_rate", ascending=False).head(20)
low_turnout_constituency = constituency_stations.sort_values("turnout_rate", ascending=True).head(20)
validation_summary = pd.concat(
    [
        create_validation_summary(constituency_stations, "constituency_stations"),
        create_validation_summary(partylist_stations, "partylist_stations"),
    ],
    ignore_index=True,
)


### Preview: top constituency candidates

In [10]:
display(candidate_summary.head(10))

,constituency,candidate_number,candidate_name,party,total_votes,avg_votes_per_station,max_votes_in_station,station_count,vote_share
7,1,8,นางทิพา ปวีณาเสถียร,พรรคประชาชน,43340,146.418919,555,296,0.397469
3,1,4,นายกิตติกร โล่ห์สุนทร,พรรคเพื่อไทย,22643,75.983221,520,298,0.207658
2,1,3,นายชวนิต จันทรสุรินทร์,พรรคภูมิใจไทย,10907,36.600671,520,298,0.100028
6,1,7,นางสาวอมลยา เจนตวนิชย์,พรรคประชาธิปัตย์,7281,24.189369,520,301,0.066774
0,1,1,นายอธิวัฒน์ ศรีไชยยานุพันธ์,พรรคคล้ายธรรม,5878,19.658863,520,299,0.053907
8,1,9,นายก้องเกียรติ มาทา,พรรครวมไทยสร้างชาติ,5774,19.182724,520,301,0.052953
5,1,6,ร.ต.อ. ประสิทธิ์ ทิวงศ์ษา,พรรคเศรษฐกิจ,5052,16.896321,520,299,0.046332
1,1,2,นายบุญเชิด พรมศร,พรรคโอกาสใหม่,4758,15.807309,520,301,0.043635
4,1,5,นายนริศ แสงบุญเรือง,พรรคไทยก้าวใหม่,3407,11.356667,520,300,0.031245
9,1,10,,,0,0.000000,0,14,0.000000


### Preview: top party-list parties

In [11]:
display(partylist_summary.head(10))

,constituency,party_number,party_name,total_votes,avg_votes_per_station,max_votes_in_station,station_count,vote_share
45,1,46,ประชาชน,37374,129.321799,575,289,0.369992
8,1,9,เพื่อไทย,20531,71.536585,188,287,0.203251
36,1,37,ภูมิใจไทย,10962,37.670103,295,291,0.108521
26,1,27,ประชาธิปัตย์,4354,14.860068,89,293,0.043103
10,1,11,เศรษฐกิจ,2818,9.650685,95,292,0.027897
1,1,2,เพื่อชาติไทย,2078,7.190311,398,289,0.020572
5,1,6,รวมไทยสร้างชาติ,1905,6.591696,29,289,0.018859
56,1,57,พลังไทยรักชาติ,1599,5.532872,443,289,0.015830
3,1,4,มิติใหม่,1400,4.827586,87,290,0.013860
7,1,8,ประชาธิปไตยใหม่,1396,4.813793,97,290,0.013820


### Preview: areas with highest turnout

In [12]:
display(constituency_area_summary.sort_values("turnout_rate", ascending=False).head(10))

,changwat,constituency,amphoe,tambon,station_count,voters_registered,voters_present,ballots_used,ballots_valid,ballots_invalid,ballots_no_vote,turnout_rate,invalid_rate,no_vote_rate
39,ลำปาง,1,หัวมิตร,นำมากล,1,7,263,263,235,15,13,37.571429,0.057034,0.04943
103,ลำปาง,1,เชียงคิตร,เชียงตาล,1,30,521,521,449,25,40,17.366667,0.047985,0.076775
48,ลำปาง,1,หัวเมืองฯ,ห้างฉัตร,1,37,637,637,543,36,58,17.216216,0.056515,0.091052
52,ลำปาง,1,หัวเสีย,บางยางคก,1,40,592,592,519,31,42,14.8,0.052365,0.070946
213,ลำปาง,1,เมืองลำปาง,ทุ่งฝาย,1,40,402,402,398,72,26,10.05,0.179104,0.064677
79,ลำปาง,1,อำเภอ/เขต ห้างสรรพสินค้า,ตำบล/แขวง/เทศบาล เมืองฯลฯ,1,47,448,448,415,18,15,9.531915,0.040179,0.033482
104,ลำปาง,1,เชียงดาว,เชียงตาล,1,47,353,353,305,18,30,7.510638,0.050992,0.084986
118,ลำปาง,1,เมือง,คลองสอง,1,21,156,156,124,12,16,7.428571,0.076923,0.102564
70,ลำปาง,1,ห้างฉัตร,แม่ลาน,1,70,512,312,512,18,26,7.314286,0.035156,0.050781
127,ลำปาง,1,เมือง,ต้นสูงชัย,1,97,698,698,607,24,67,7.195876,0.034384,0.095989


### Preview: areas with highest invalid ballot rate

In [13]:
display(constituency_area_summary.sort_values("invalid_rate", ascending=False).head(10))

,changwat,constituency,amphoe,tambon,station_count,voters_registered,voters_present,ballots_used,ballots_valid,ballots_invalid,ballots_no_vote,turnout_rate,invalid_rate,no_vote_rate
174,ลำปาง,1,เมือง,สมชัย,1,526,344,344,290,8715,39,0.653992,25.334302,0.113372
222,ลำปาง,1,เมืองลำปาง,บ้านค่ำ,1,418,1,273,233,19,21,0.002392,19.0,21.0
72,ลำปาง,1,ห้าชนิด,เมืองฮาว,1,43,34,371,328,158,56,0.790698,4.647059,1.647059
141,ลำปาง,1,เมือง,บ้านเชื่อม,2,434,321,771,855,514,511,0.739631,1.601246,1.5919
248,ลำปาง,1,เมืองฯ,นิคมพัฒนา,1,239,185,185,173,173,6,0.774059,0.935135,0.032432
36,ลำปาง,1,หนองคู,มอญปาก,1,100,400,400,372,372,4,4.0,0.93,0.01
236,ลำปาง,1,เมืองลำปาง,หัวเวียง,1,452,277,277,245,243,3,0.612832,0.877256,0.01083
130,ลำปาง,1,เมือง,ทุ่งลาย,1,539,406,406,339,339,34,0.753247,0.834975,0.083744
55,ลำปาง,1,หาดใหญ่,เชิงคาม,1,297,47,494,425,36,206,0.158249,0.765957,4.382979
10,ลำปาง,1,ซังฮัตร,เชียงตาล,1,40,260,260,230,176,14,6.5,0.676923,0.053846


### Preview: most competitive constituency polling stations

In [14]:
display(candidate_competitive_stations.head(10))

rank_in_station,sha256,changwat,constituency,polling_station,amphoe,tambon,rank1_votes,rank2_votes,vote_margin,margin_rate
150,90e3b76127a1185f63dfa082b04a3b223f80c0911e4477...,ลำปาง,1,6,เมือง,เวียงเหนือ,0,0,0,<NA>
28,1545296537816b1b23074f2fabb67f1def78e05c59e64d...,ลำปาง,1,3,น้ำผึ้ง,ตำบล/แขวง/เทศบาล 20,520,520,0,0.0
123,720c001ba97d9faec9b2d4fb0bfcc7723ca085e932319c...,ลำปาง,1,24,เชิงเมือง,หัวเวียง,259,259,0,0.0
120,6c7f86676fedd26b8329784d7de803f103e595efc238f0...,ลำปาง,1,10,เมือง,นิคมพัฒนา,0,0,0,<NA>
114,65c410a07494decd3a70d7b7036f193afb37dec6bc0d52...,ลำปาง,1,4,เพิ่มลำปาง,ป่านคำ,0,0,0,<NA>
76,47450f13216126e0d3bbeb2b9265713ab938b7363c7b53...,ลำปาง,1,2,ห้างฉัตร,วอแก้ว,60,60,0,0.0
193,b51359796f2db407ef021ab13cd3752f01d6973d478084...,ลำปาง,1,2,เมืองลำปาง,หนองคาย,0,0,0,<NA>
22,11ae4f7d0e5d309a9587bdc34d701ad66943da25c4c8d5...,ลำปาง,1,4,เมืองลำปาง,ทุ่มฟาย,0,0,0,<NA>
294,f7f5874cdadc00284fd2dad3f3108e3902883c1eadc078...,ลำปาง,1,14,เมือง,นิคมพัฒนา,0,0,0,<NA>
110,5e61b82fe0143795b691c26d2c7ee499b4730f1da22032...,ลำปาง,1,74,เมืองฯ,ธมฟ,0,0,0,<NA>


## 6. Export analysis CSVs

These files are useful for Streamlit or dashboard visualization.

In [15]:
outputs = {
    "constituency_station_metrics": constituency_stations,
    "partylist_station_metrics": partylist_stations,
    "candidate_summary": candidate_summary,
    "partylist_summary": partylist_summary,
    "constituency_area_summary": constituency_area_summary,
    "partylist_area_summary": partylist_area_summary,
    "candidate_vote_station_joined": candidate_joined,
    "partylist_vote_station_joined": partylist_joined,
    "candidate_winner_by_station": candidate_winner_by_station,
    "partylist_winner_by_station": partylist_winner_by_station,
    "candidate_competitive_stations": candidate_competitive_stations,
    "partylist_competitive_stations": partylist_competitive_stations,
    "high_invalid_constituency_stations": high_invalid_constituency,
    "low_turnout_constituency_stations": low_turnout_constituency,
    "validation_summary": validation_summary,
}

OUT_DIR.mkdir(parents=True, exist_ok=True)

for name, df in outputs.items():
    if df is None or df.empty:
        print(f"Skipped empty output: {name}")
        continue
    path = OUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved: {path}")

print("\nExport:", OUT_DIR)


Saved: /Users/wutthichod/Documents/dsde/final_project/data/analysis_exports/constituency_station_metrics.csv
Saved: /Users/wutthichod/Documents/dsde/final_project/data/analysis_exports/partylist_station_metrics.csv
Saved: /Users/wutthichod/Documents/dsde/final_project/data/analysis_exports/candidate_summary.csv
Saved: /Users/wutthichod/Documents/dsde/final_project/data/analysis_exports/partylist_summary.csv
Saved: /Users/wutthichod/Documents/dsde/final_project/data/analysis_exports/constituency_area_summary.csv
Saved: /Users/wutthichod/Documents/dsde/final_project/data/analysis_exports/partylist_area_summary.csv
Saved: /Users/wutthichod/Documents/dsde/final_project/data/analysis_exports/candidate_vote_station_joined.csv
Saved: /Users/wutthichod/Documents/dsde/final_project/data/analysis_exports/partylist_vote_station_joined.csv
Saved: /Users/wutthichod/Documents/dsde/final_project/data/analysis_exports/candidate_winner_by_station.csv
Saved: /Users/wutthichod/Documents/dsde/final_projec